# Ledger | Project 15
Group 4 · Domain-Specific Document Classifier

Run this notebook from the project directory.
It examines the shipped data splits, reproduces the evaluation from saved predictions, and runs the deployed classifier.

[CFPB source](https://www.consumerfinance.gov/data-research/consumer-complaints/) · [Book extract](https://smltar.com/appendixdata) · [BERT-Tiny](https://huggingface.co/google/bert_uncased_L-2_H-128_A-2)

In [ ]:
import csv
import json
from collections import Counter
from pathlib import Path
from statistics import median

from ledger import Classifier, LABELS, words
from train import metrics

def load_csv(path):
    with open(path, encoding='utf-8', newline='') as stream:
        return list(csv.DictReader(stream))

rows = load_csv('data/splits.csv')
report = json.loads(Path('analysis/metrics.json').read_text(encoding='utf-8'))
print(f'Cleaned complaints: {len(rows):,}')
assert len({row['id'] for row in rows}) == len(rows)
assert len({' '.join(words(row['text'])) for row in rows}) == len(rows)

## Class balance and length
The published archive is imbalanced.
This experiment limits each class to 450 cleaned complaints and stratifies the split.
The table compares the original eligible counts with split counts and shows median length after cleaning.

In [ ]:
source = json.loads(Path('data/source.json').read_text(encoding='utf-8'))
counts = Counter((row['label'], row['split']) for row in rows)
print(f"{'Category':20} {'Available':>10} {'Train':>8} {'Val':>8} {'Test':>8} {'Median words':>13}")
for label in LABELS:
    sizes = [counts[label, split] for split in ('train', 'validation', 'test')]
    length = median(len(words(row['text'])) for row in rows if row['label'] == label)
    print(f"{label:20} {source['eligible_by_label'][label]:>10,} {sizes[0]:>8} {sizes[1]:>8} {sizes[2]:>8} {length:>13.0f}")
print('Cleaning audit:', report['data']['cleaning'])
print('BERT-truncated complaints:', report['data']['truncated'])

## Feature engineering and training
The lexical baseline uses normalized TF-IDF word unigrams and bigrams with multinomial Naive Bayes.
The transformer uses WordPiece IDs and updates its final output projection and LayerNorm with a new classifier.
The ensemble mixes temperature-scaled logits and chooses the mix on validation data.

To rerun training, use `python collect.py --weights` and `python train.py --output .build/reproduction` in the project directory.

In [ ]:
training = report['training']
print('Selected epoch:', training['selected_epoch'])
print('Final encoder projection L2 change:', round(training['encoder_projection_delta_l2'], 4))
print('BERT weight:', f"{training['bert_weight']:.0%}")
for item in training['epochs']:
    print(f"Epoch {item['epoch']:2d}  val macro F1 {item['validation_f1']:.3f}")

## Held-out evaluation
The test set was held out during training and calibration.
Check accuracy, macro precision, macro recall, and macro F1, followed by the ensemble confusion matrix.

In [ ]:
import numpy as np

predictions = load_csv('analysis/test_predictions.csv')
actual = np.array([LABELS.index(row['actual']) for row in predictions])
print(f"{'Model':13} {'Accuracy':>9} {'Precision':>10} {'Recall':>8} {'Macro F1':>9}")
for key in ('bert', 'baseline', 'ensemble'):
    predicted = np.array([LABELS.index(row[key]) for row in predictions])
    current = metrics(actual, predicted)
    assert current == report[key]
    print(f"{key:13} {current['accuracy']:>9.1%} {current['macro_precision']:>10.1%} "
          f"{current['macro_recall']:>8.1%} {current['macro_f1']:>9.1%}")
print('Rows are true labels, columns are predicted labels:', LABELS)
print(np.array(report['ensemble']['confusion_matrix']))

## Inference
This is the same classifier the web app calls.
The examples are fictional.
BERT reads up to 126 content tokens, while TF-IDF uses all extracted text.

In [ ]:
classifier = Classifier()
for example in sorted(Path('examples').glob('*.txt')):
    result = classifier.predict(example.read_text(encoding='utf-8'))
    print(f"{example.name}: {result['category']} ({result['confidence']:.1%})")